# Lab 5A: Adversarial Search, Game Theory, and Minimax

## Learning goals
By the end of this lab, students can:
- define a two-player, turn-taking, deterministic, perfect-information, zero-sum game;
- implement the Chapter 5 game interface: `to_move`, `actions`, `result`, `is_terminal`, and `utility`;
- compute minimax values for a complete game tree;
- visually step through the recursive search and value backup process;
- play against a minimax Tic-Tac-Toe agent.

Source note: These notes and examples are aligned with Chapter 5, *Adversarial Search and Games*. The code is self-contained and uses only local Python, matplotlib, and optional ipywidgets.

## Chapter 5 notes for this lab

A game can be modeled as a search problem with adversaries. A state tells us the current position; `TO-MOVE(s)` tells us whose turn it is; `ACTIONS(s)` lists legal moves; `RESULT(s, a)` applies a move; `IS-TERMINAL(s)` stops at wins, losses, or draws; and `UTILITY(s, p)` gives a numeric outcome for a player.

In a two-player zero-sum game, MAX wants high utility and MIN wants low utility. Minimax assumes both players play optimally:

\[
MINIMAX(s)=
\begin{cases}
UTILITY(s) & \text{if terminal}\\
\max_a MINIMAX(RESULT(s,a)) & \text{if MAX to move}\\
\min_a MINIMAX(RESULT(s,a)) & \text{if MIN to move}
\end{cases}
\]

The visualization below lets students see the depth-first recursion, leaf utilities, and backed-up minimax values.

In [ ]:
# Run this cell first.
# These notebooks use local Python code plus matplotlib. ipywidgets is optional.
# If a student machine is missing packages, uncomment and run once:
# %pip install matplotlib ipywidgets

import math
from functools import lru_cache
from collections import defaultdict

import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle, Circle

from IPython.display import display, HTML, clear_output

try:
    import ipywidgets as widgets
    WIDGETS_AVAILABLE = True
except Exception:
    WIDGETS_AVAILABLE = False
    print("ipywidgets is not available. Static first/last frames will still work.")

MAX_PLAYER = "X"
MIN_PLAYER = "O"
EMPTY = " "


def make_stepper(steps, draw_func, info_func=None, title="Search stepper"):
    """Small GUI stepper. Falls back to first/last static frames without ipywidgets."""
    if not steps:
        print("No steps to display.")
        return
    if WIDGETS_AVAILABLE:
        slider = widgets.IntSlider(value=0, min=0, max=len(steps)-1, step=1, description="step")
        play = widgets.Play(value=0, min=0, max=len(steps)-1, step=1, interval=700, description="play")
        widgets.jslink((play, "value"), (slider, "value"))
        out = widgets.Output()
        def render(change=None):
            with out:
                clear_output(wait=True)
                display(HTML(f"<h3>{title}</h3>"))
                if info_func:
                    display(HTML(info_func(steps[slider.value], slider.value)))
                draw_func(steps[slider.value], slider.value)
                plt.show()
        slider.observe(render, names="value")
        display(widgets.VBox([widgets.HBox([play, slider]), out]))
        render()
    else:
        for i in [0, len(steps)-1]:
            if info_func:
                display(HTML(info_func(steps[i], i)))
            draw_func(steps[i], i)
            plt.show()

In [ ]:
WIN_LINES = [
    (0, 1, 2), (3, 4, 5), (6, 7, 8),
    (0, 3, 6), (1, 4, 7), (2, 5, 8),
    (0, 4, 8), (2, 4, 6),
]

MOVE_NAMES = {
    0: "upper-left", 1: "upper-middle", 2: "upper-right",
    3: "middle-left", 4: "center", 5: "middle-right",
    6: "lower-left", 7: "lower-middle", 8: "lower-right",
}

class TicTacToe:
    """Chapter 5 game object for Tic-Tac-Toe."""
    def __init__(self):
        self.initial = (EMPTY,) * 9

    def to_move(self, state):
        return MAX_PLAYER if state.count(MAX_PLAYER) == state.count(MIN_PLAYER) else MIN_PLAYER

    def actions(self, state):
        # Good pedagogical move ordering: center, corners, then edges.
        order = [4, 0, 2, 6, 8, 1, 3, 5, 7]
        return [i for i in order if state[i] == EMPTY]

    def result(self, state, action):
        if state[action] != EMPTY:
            raise ValueError("Illegal move: square is not empty.")
        mark = self.to_move(state)
        new_state = list(state)
        new_state[action] = mark
        return tuple(new_state)

    def winner(self, state):
        for a, b, c in WIN_LINES:
            if state[a] != EMPTY and state[a] == state[b] == state[c]:
                return state[a]
        return None

    def is_terminal(self, state):
        return self.winner(state) is not None or EMPTY not in state

    def utility(self, state):
        """Utility from MAX/X viewpoint: +1 win, -1 loss, 0 draw."""
        w = self.winner(state)
        if w == MAX_PLAYER:
            return 1
        if w == MIN_PLAYER:
            return -1
        return 0


game = TicTacToe()

def state_from_rows(rows):
    """Use '.' for empty cells: state_from_rows(['XO.', '.X.', 'O..'])."""
    return tuple(c if c in "XO" else EMPTY for row in rows for c in row)


def draw_ttt_board(ax, state, title="", last_move=None, winning_line=None):
    ax.set_xlim(0, 3)
    ax.set_ylim(0, 3)
    ax.set_aspect("equal")
    ax.axis("off")
    ax.set_title(title)
    for i in range(4):
        ax.plot([0, 3], [i, i], color="black", lw=2)
        ax.plot([i, i], [0, 3], color="black", lw=2)
    if last_move is not None:
        r, c = divmod(last_move, 3)
        ax.add_patch(Rectangle((c, 2-r), 1, 1, facecolor="#fff2aa", edgecolor="none", alpha=0.7))
    for idx, mark in enumerate(state):
        if mark == EMPTY:
            continue
        r, c = divmod(idx, 3)
        x, y = c + 0.5, 2 - r + 0.5
        if mark == MAX_PLAYER:
            ax.text(x, y, "X", ha="center", va="center", fontsize=30, fontweight="bold", color="#1f77b4")
        else:
            ax.text(x, y, "O", ha="center", va="center", fontsize=30, fontweight="bold", color="#d62728")
    if winning_line:
        pts = []
        for idx in winning_line:
            r, c = divmod(idx, 3)
            pts.append((c+0.5, 2-r+0.5))
        ax.plot([pts[0][0], pts[-1][0]], [pts[0][1], pts[-1][1]], color="green", lw=5, alpha=0.6)


def board_text(state):
    rows = []
    for r in range(3):
        rows.append("".join(state[3*r:3*r+3]).replace(" ", "."))
    return "/".join(rows)

In [ ]:
@lru_cache(maxsize=None)
def minimax_value(state):
    """Return the backed-up minimax value from the MAX/X viewpoint."""
    if game.is_terminal(state):
        return game.utility(state)
    player = game.to_move(state)
    child_values = [minimax_value(game.result(state, action)) for action in game.actions(state)]
    return max(child_values) if player == MAX_PLAYER else min(child_values)


def minimax_decision(state):
    """Return best action and value for the player to move."""
    player = game.to_move(state)
    scored = [(action, minimax_value(game.result(state, action))) for action in game.actions(state)]
    if player == MAX_PLAYER:
        return max(scored, key=lambda x: x[1])
    return min(scored, key=lambda x: x[1])


def minimax_events(root_state):
    """Trace recursive minimax so students can watch the search unfold."""
    events = []
    def search(state, depth=0, path=()):
        player = game.to_move(state) if not game.is_terminal(state) else "terminal"
        events.append({
            "kind": "enter", "state": state, "path": path, "depth": depth,
            "message": f"Enter node {board_text(state)} at depth {depth}. To move: {player}."
        })
        if game.is_terminal(state):
            value = game.utility(state)
            events.append({
                "kind": "leaf", "state": state, "path": path, "depth": depth, "value": value,
                "message": f"Terminal state reached. Utility for MAX/X = {value}."
            })
            return value
        player = game.to_move(state)
        best_value = -math.inf if player == MAX_PLAYER else math.inf
        best_action = None
        for action in game.actions(state):
            child = game.result(state, action)
            events.append({
                "kind": "try", "state": state, "child": child, "path": path,
                "child_path": path + (action,), "action": action, "depth": depth,
                "message": f"Try {player} move {action} ({MOVE_NAMES[action]})."
            })
            value = search(child, depth + 1, path + (action,))
            if player == MAX_PLAYER and value > best_value:
                best_value, best_action = value, action
            if player == MIN_PLAYER and value < best_value:
                best_value, best_action = value, action
            rule = "maximum" if player == MAX_PLAYER else "minimum"
            events.append({
                "kind": "backup", "state": state, "path": path, "depth": depth,
                "value": best_value, "best_action": best_action,
                "message": f"Back up value {best_value}. {player} keeps the {rule} child so far: square {best_action}."
            })
        events.append({
            "kind": "return", "state": state, "path": path, "depth": depth,
            "value": best_value, "best_action": best_action,
            "message": f"Return minimax value {best_value} from node {board_text(state)}."
        })
        return best_value
    search(root_state)
    return events

In [ ]:
def build_visual_tree(root_state, max_depth=3):
    nodes, edges = [], []
    def rec(state, path, depth, parent_id=None, action=None):
        node_id = len(nodes)
        nodes.append({"id": node_id, "state": state, "path": path, "depth": depth, "action": action})
        if parent_id is not None:
            edges.append((parent_id, node_id, action))
        if depth < max_depth and not game.is_terminal(state):
            for a in game.actions(state):
                rec(game.result(state, a), path + (a,), depth + 1, node_id, a)
        return node_id
    rec(root_state, (), 0)
    children = defaultdict(list)
    for p, c, a in edges:
        children[p].append(c)
    x_counter = [0]
    pos = {}
    def assign(nid):
        if not children[nid]:
            x = x_counter[0]
            x_counter[0] += 1
        else:
            xs = [assign(ch) for ch in children[nid]]
            x = sum(xs) / len(xs)
        pos[nid] = (x, -nodes[nid]["depth"])
        return x
    assign(0)
    return nodes, edges, pos


def draw_search_tree(ax, root_state, event_index, events, max_depth=3):
    nodes, edges, pos = build_visual_tree(root_state, max_depth=max_depth)
    prefix = events[:event_index+1]
    seen = set()
    returned = {}
    for ev in prefix:
        if "path" in ev:
            seen.add(ev["path"])
        if "child_path" in ev:
            seen.add(ev["child_path"])
        if ev["kind"] in ("leaf", "return"):
            returned[ev["path"]] = ev.get("value")
    current_path = events[event_index].get("child_path", events[event_index].get("path", ()))
    node_by_id = {n["id"]: n for n in nodes}
    for p, c, action in edges:
        x1, y1 = pos[p]
        x2, y2 = pos[c]
        child_path = node_by_id[c]["path"]
        active = child_path in seen
        ax.plot([x1, x2], [y1, y2], color="#555555" if active else "#cccccc", lw=1.6 if active else 0.8)
        ax.text((x1+x2)/2, (y1+y2)/2 + 0.05, str(action), fontsize=8, ha="center", color="#555555")
    for n in nodes:
        x, y = pos[n["id"]]
        path = n["path"]
        if path == current_path:
            face, edge, lw = "#ffbf66", "#222222", 2.5
        elif path in returned:
            face, edge, lw = "#b6e3a8", "#333333", 1.5
        elif path in seen:
            face, edge, lw = "#d7e9ff", "#333333", 1.2
        else:
            face, edge, lw = "#eeeeee", "#999999", 0.8
        ax.add_patch(Circle((x, y), 0.22, facecolor=face, edgecolor=edge, lw=lw))
        if path == ():
            label = "root"
        else:
            label = str(path[-1])
        if path in returned:
            label += f"\nv={returned[path]:+g}"
        ax.text(x, y, label, ha="center", va="center", fontsize=8)
    ax.set_title(f"Game tree, first {max_depth} ply shown")
    ax.axis("off")
    xs = [p[0] for p in pos.values()]
    ys = [p[1] for p in pos.values()]
    ax.set_xlim(min(xs)-0.8, max(xs)+0.8)
    ax.set_ylim(min(ys)-0.6, max(ys)+0.6)


def root_score_table(root_state):
    rows = []
    for action in game.actions(root_state):
        child = game.result(root_state, action)
        rows.append((action, MOVE_NAMES[action], minimax_value(child)))
    return rows


def draw_root_scores(ax, root_state, current_action=None):
    rows = root_score_table(root_state)
    actions = [r[0] for r in rows]
    values = [r[2] for r in rows]
    colors = ["#ffbf66" if a == current_action else "#8ecae6" for a in actions]
    ax.bar([str(a) for a in actions], values, color=colors, edgecolor="black")
    ax.axhline(0, color="black", lw=0.8)
    ax.set_ylim(-1.2, 1.2)
    ax.set_xlabel("root move")
    ax.set_ylabel("minimax value")
    ax.set_title("Root move values")
    for i, v in enumerate(values):
        ax.text(i, v + (0.08 if v >= 0 else -0.15), f"{v:+g}", ha="center", fontsize=9)


def draw_minimax_snapshot(ev, idx):
    root_state = DEMO_STATE
    current_state = ev.get("child", ev.get("state", root_state))
    current_action = ev.get("action", None)
    fig = plt.figure(figsize=(15, 6))
    gs = fig.add_gridspec(1, 3, width_ratios=[1.0, 2.7, 1.1])
    ax_board = fig.add_subplot(gs[0, 0])
    ax_tree = fig.add_subplot(gs[0, 1])
    ax_scores = fig.add_subplot(gs[0, 2])
    title = f"Current node: {board_text(current_state)}"
    draw_ttt_board(ax_board, current_state, title=title, last_move=current_action)
    draw_search_tree(ax_tree, root_state, idx, EVENTS, max_depth=3)
    draw_root_scores(ax_scores, root_state, current_action=current_action)
    fig.tight_layout()


def minimax_info_html(ev, idx):
    return f"""
    <div style='font-family:Arial;line-height:1.35'>
    <b>Step {idx+1} of {len(EVENTS)}</b> &nbsp; <b>Event:</b> {ev['kind']}<br>
    {ev['message']}<br>
    <span style='color:#1f77b4'><b>X is MAX</b></span>; <span style='color:#d62728'><b>O is MIN</b></span>.
    </div>
    """

In [ ]:
# Demonstration board: X is MAX and has a tactical winning move at square 8.
DEMO_STATE = state_from_rows([
    "XO.",
    ".X.",
    "O..",
])

print("Initial demo board:", board_text(DEMO_STATE))
print("Player to move:", game.to_move(DEMO_STATE))
print("Best minimax decision:", minimax_decision(DEMO_STATE))
print("Root scores:")
for action, name, value in root_score_table(DEMO_STATE):
    print(f"  square {action:>1} ({name:>12}): {value:+g}")

EVENTS = minimax_events(DEMO_STATE)
make_stepper(EVENTS, draw_minimax_snapshot, minimax_info_html, title="Minimax recursive search and value backup")

## Optional GUI: play against the minimax agent

This is a complete local game. Students can play as X; the agent replies as O using minimax. Because Tic-Tac-Toe is solved, optimal play by both sides ends in a draw.

In [ ]:
def play_against_minimax():
    if not WIDGETS_AVAILABLE:
        print("Install ipywidgets to use the clickable game board: pip install ipywidgets")
        return
    state = [EMPTY] * 9
    buttons = [widgets.Button(description=" ", layout=widgets.Layout(width="70px", height="70px")) for _ in range(9)]
    status = widgets.HTML("<b>You are X. Choose a square.</b>")
    out = widgets.Output()

    def refresh():
        for i, b in enumerate(buttons):
            b.description = state[i] if state[i] != EMPTY else " "
            b.disabled = state[i] != EMPTY or game.is_terminal(tuple(state))
        with out:
            clear_output(wait=True)
            fig, ax = plt.subplots(figsize=(3.2, 3.2))
            draw_ttt_board(ax, tuple(state), title="Human X vs minimax O")
            plt.show()

    def agent_move():
        s = tuple(state)
        if game.is_terminal(s):
            return
        action, value = minimax_decision(s)
        state[action] = MIN_PLAYER
        status.value = f"Agent O moved at square {action}. Predicted value for X: {value:+g}."

    def click(i):
        def handler(_):
            if state[i] != EMPTY or game.is_terminal(tuple(state)):
                return
            state[i] = MAX_PLAYER
            s = tuple(state)
            if not game.is_terminal(s):
                agent_move()
            w = game.winner(tuple(state))
            if w:
                status.value += f"<br><b>Winner: {w}</b>"
            elif game.is_terminal(tuple(state)):
                status.value += "<br><b>Draw.</b>"
            refresh()
        return handler

    for i, b in enumerate(buttons):
        b.on_click(click(i))
    grid = widgets.GridBox(buttons, layout=widgets.Layout(grid_template_columns="repeat(3, 75px)"))
    display(widgets.VBox([status, grid, out]))
    refresh()

play_against_minimax()

## Student practice

Change `STUDENT_STATE` to a different legal Tic-Tac-Toe state and rerun the cell. Suggested questions:
1. Which square is optimal for the player to move?
2. Which leaves determine the backed-up value?
3. Does the best-looking immediate move always match the minimax move?

In [ ]:
STUDENT_STATE = state_from_rows([
    "X..",
    ".O.",
    "..X",
])

print("Student board:", board_text(STUDENT_STATE))
print("Player to move:", game.to_move(STUDENT_STATE))
print("Best minimax decision:", minimax_decision(STUDENT_STATE))

# Uncomment to step through the student's board.
# EVENTS = minimax_events(STUDENT_STATE)
# DEMO_STATE = STUDENT_STATE
# make_stepper(EVENTS, draw_minimax_snapshot, minimax_info_html, title="Student minimax trace")